# ðŸ”¬ Sprint 2 â€” Medical Research Agent
**Project:** Autonomous Medical Research & Patient Triage Assistant  
**Day:** 2 of 5 | **Sprint Goal:** Research symptoms from PubMed + WHO/NHIS guidelines â†’ store in Pinecone

## What we build today
```
Symptoms â†’ PubMed API Search â†’ WHO/NHIS Web Search â†’ OpenAI Synthesis
         â†’ Chunk Text â†’ Embed (text-embedding-3-small) â†’ Upsert to Pinecone
```

## Sprint 2 User Story
> **US-03:** As a patient, the agent researches my symptoms against WHO and NHIS Nigeria guidelines,  
> with evidence from peer-reviewed PubMed literature.

**Definition of Done:** Pinecone index contains â‰¥10 clinical vectors per symptom query  
with correct metadata (source, condition, evidence_level)

## âš ï¸ Clinical Safety Rule
> All research must be grounded in verified sources:  
> **PubMed** (peer-reviewed) + **WHO** (international guidelines) + **NHIS Nigeria** (local protocols)  
> The agent must NEVER fabricate clinical information.

## 1. Imports & Setup

In [23]:
import os, json, time, uuid, re, requests
from datetime import datetime, timezone
from typing import TypedDict, Optional, List, Dict, Any
from dotenv import load_dotenv

from langgraph.graph import StateGraph, END
from openai import OpenAI
from pinecone import Pinecone, ServerlessSpec

load_dotenv()
OPENAI_API_KEY   = os.getenv('OPENAI_API_KEY', '')
PINECONE_API_KEY = os.getenv('PINECONE_API_KEY', '')

assert OPENAI_API_KEY,   'âŒ Set OPENAI_API_KEY'
assert PINECONE_API_KEY, 'âŒ Set PINECONE_API_KEY'

openai_client = OpenAI(api_key=OPENAI_API_KEY)
pc            = Pinecone(api_key=PINECONE_API_KEY)

# â”€â”€ Config â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
PINECONE_INDEX_NAME = 'medical-triage-agent'   # Separate index from research agent
PINECONE_INDEX_HOST = os.getenv('PINECONE_INDEX_HOST', '').strip()
EMBED_MODEL         = 'text-embedding-3-small'
EMBED_DIMS          = 1536
RESEARCH_MODEL      = 'gpt-4o-mini'
PUBMED_BASE_URL     = 'https://eutils.ncbi.nlm.nih.gov/entrez/eutils'

print('âœ… All clients initialised')
print(f'   Pinecone index : {PINECONE_INDEX_NAME}')
print(f'   Embed model    : {EMBED_MODEL}')
print(f'   Research model : {RESEARCH_MODEL}')

âœ… All clients initialised
   Pinecone index : medical-triage-agent
   Embed model    : text-embedding-3-small
   Research model : gpt-4o-mini


## 2. MedicalAgentState Schema (from Sprint 1)

In [24]:
class MedicalAgentState(TypedDict):
    session_id:        str
    ticket_id:         str
    initiated_at:      str
    channel:           str
    chat_id:           str
    user_mode:         str
    raw_message:       str
    symptoms:          List[str]
    duration:          Optional[str]
    age:               Optional[str]
    medications:       List[str]
    status:            str
    errors:            List[str]
    raw_research:      Optional[str]
    research_chunks:   Optional[List[str]]
    pinecone_ids:      Optional[List[str]]
    clinical_namespace: Optional[str]
    retrieved_chunks:  Optional[List[str]]
    reranked_chunks:   Optional[List[str]]
    urgency_level:     Optional[str]
    urgency_score:     Optional[int]
    differential:      Optional[List[str]]
    red_flags:         Optional[List[str]]
    retry_count:       int
    triage_report:     Optional[Dict[str, str]]
    nutrition_advice:  Optional[str]
    home_remedies:     Optional[str]
    follow_up_sent:    bool
    report_ready:      bool
    workflow_path:     List[str]

def create_medical_state(message: str, symptoms: List[str] = None,
                          mode: str = 'patient', channel: str = 'telegram') -> MedicalAgentState:
    ticket = f'MED-{datetime.now(timezone.utc).strftime("%Y%m%d")}-{str(uuid.uuid4())[:8].upper()}'
    return MedicalAgentState(
        session_id=str(uuid.uuid4()), ticket_id=ticket,
        initiated_at=datetime.now(timezone.utc).isoformat(),
        channel=channel, chat_id='test-chat',
        user_mode=mode, raw_message=message,
        symptoms=symptoms or [message],
        duration=None, age=None, medications=[],
        status='acknowledged', errors=[],
        raw_research=None, research_chunks=None, pinecone_ids=None,
        clinical_namespace=None,
        retrieved_chunks=None, reranked_chunks=None,
        urgency_level=None, urgency_score=None,
        differential=None, red_flags=[],
        retry_count=0,
        triage_report=None, nutrition_advice=None,
        home_remedies=None, follow_up_sent=False,
        report_ready=False, workflow_path=[]
    )

print('âœ… MedicalAgentState schema ready')

âœ… MedicalAgentState schema ready


## 3. Pinecone Index Setup

In [25]:
def _normalize_pinecone_host(host: Optional[str]) -> Optional[str]:
    host = (host or '').strip()
    if not host:
        return None
    # Pinecone needs the full data-plane host, not the index name.
    if 'pinecone.io' not in host:
        return None
    return host

def setup_pinecone_index(index_name: str, dims: int = 1536, max_attempts: int = 5):
    last_error = None
    for attempt in range(1, max_attempts + 1):
        try:
            existing = [idx.name for idx in pc.list_indexes()]
            if index_name not in existing:
                print(f'[PINECONE] Creating index "{index_name}" ({dims} dims)...')
                pc.create_index(
                    name=index_name, dimension=dims, metric='cosine',
                    spec=ServerlessSpec(cloud='aws', region='us-east-1')
                )

            while True:
                desc = pc.describe_index(index_name)
                if desc.status['ready']:
                    host = _normalize_pinecone_host(getattr(desc, 'host', None))
                    if not host and isinstance(desc, dict):
                        host = _normalize_pinecone_host(desc.get('host'))
                    if not host:
                        host = _normalize_pinecone_host(PINECONE_INDEX_HOST)
                    if not host:
                        raise RuntimeError(
                            f"Pinecone index {index_name!r} is ready, but no host was returned. "
                            'Set PINECONE_INDEX_HOST to the full Pinecone data-plane host '
                            '(for example, a host ending in .pinecone.io).'
                        )
                    print(f'[PINECONE] Index ready: {index_name} ({host})')
                    return pc.Index(host=host)
                print('  Waiting...')
                time.sleep(2)
        except Exception as e:
            last_error = e
            if attempt == max_attempts:
                raise RuntimeError(
                    f'Failed to initialize Pinecone index {index_name!r} after {max_attempts} attempts. '
                    'This usually means a transient network/SSL problem or a blocked outbound connection.'
                ) from e
            delay = min(2 ** attempt, 15)
            print(f'[PINECONE] Setup attempt {attempt}/{max_attempts} failed: {e}. Retrying in {delay}s...')
            time.sleep(delay)

    raise RuntimeError(f'Failed to initialize Pinecone index {index_name!r}: {last_error}')

pinecone_index = setup_pinecone_index(PINECONE_INDEX_NAME, EMBED_DIMS)
print(f'[PINECONE] Stats: {pinecone_index.describe_index_stats()}')

[PINECONE] âœ… Index "medical-triage-agent" already exists
[PINECONE] Stats: DescribeIndexStatsResponse(dimension=1536, total_vector_count=48, metric='cosine', namespaces=2)


## 4. PubMed API Integration

In [26]:
def search_pubmed(query: str, max_results: int = 5) -> List[Dict]:
    """
    Search PubMed for peer-reviewed medical literature.
    Uses NCBI E-utilities API (free, no API key required for basic use).
    
    Args:
        query: Medical search query
        max_results: Number of articles to retrieve
    
    Returns:
        List of article dicts with title, abstract, authors, year
    """
    articles = []

    try:
        # Step 1: Search for article IDs
        search_params = {
            'db':      'pubmed',
            'term':    f'{query}[Title/Abstract] AND (Nigeria OR Africa OR "sub-saharan")[Title/Abstract]',
            'retmax':  max_results,
            'retmode': 'json',
            'sort':    'relevance'
        }
        search_resp = requests.get(
            f'{PUBMED_BASE_URL}/esearch.fcgi',
            params=search_params, timeout=10
        )
        search_data = search_resp.json()
        ids = search_data.get('esearchresult', {}).get('idlist', [])

        if not ids:
            # Broaden search without Nigeria filter
            search_params['term'] = f'{query}[Title/Abstract]'
            search_resp = requests.get(
                f'{PUBMED_BASE_URL}/esearch.fcgi',
                params=search_params, timeout=10
            )
            search_data = search_resp.json()
            ids = search_data.get('esearchresult', {}).get('idlist', [])

        if not ids:
            print(f'[PUBMED] No results for: {query}')
            return []

        print(f'[PUBMED] Found {len(ids)} articles for: {query[:50]}')

        # Step 2: Fetch article summaries
        fetch_params = {
            'db':      'pubmed',
            'id':      ','.join(ids),
            'retmode': 'json',
            'rettype': 'abstract'
        }
        fetch_resp = requests.get(
            f'{PUBMED_BASE_URL}/esummary.fcgi',
            params=fetch_params, timeout=10
        )
        fetch_data = fetch_resp.json()
        result     = fetch_data.get('result', {})

        for pmid in ids:
            article = result.get(pmid, {})
            if article:
                articles.append({
                    'pmid':    pmid,
                    'title':   article.get('title', 'Unknown'),
                    'authors': ', '.join([a.get('name', '') for a in article.get('authors', [])[:3]]),
                    'year':    article.get('pubdate', 'Unknown')[:4],
                    'journal': article.get('source', 'Unknown'),
                    'url':     f'https://pubmed.ncbi.nlm.nih.gov/{pmid}/',
                    'source':  'pubmed'
                })

    except Exception as e:
        print(f'[PUBMED] âš ï¸ Error: {e}')

    return articles


# â”€â”€ Test PubMed â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
print('Testing PubMed API...')
test_articles = search_pubmed('malaria treatment Nigeria', max_results=3)
print(f'\nFound {len(test_articles)} articles:')
for a in test_articles:
    print(f'  ðŸ“„ [{a["year"]}] {a["title"][:70]}...')
    print(f'       Authors: {a["authors"][:50]}')
    print(f'       URL: {a["url"]}')

Testing PubMed API...
[PUBMED] Found 3 articles for: malaria treatment Nigeria

Found 3 articles:
  ðŸ“„ [2023] The impact of community delivery of intermittent preventive treatment ...
       Authors: GonzÃ¡lez R, Manun'Ebo MF, Meremikwu M
       URL: https://pubmed.ncbi.nlm.nih.gov/36925177/
  ðŸ“„ [2023] Drug Use Practices and Self-Treatment for Suspected Malaria in Ibadan,...
       Authors: Bamikole OJ, Olajide TH, Adedeji BA
       URL: https://pubmed.ncbi.nlm.nih.gov/37068754/
  ðŸ“„ [2025] Prevalence, characteristics, and treatment outcome of congenital malar...
       Authors: Kokori E, Olatunji G, Ukoaka BM
       URL: https://pubmed.ncbi.nlm.nih.gov/39844157/


## 5. Medical Research Topics

In [27]:
# Research topics for each symptom presentation
# Each generates a separate OpenAI web search query
MEDICAL_RESEARCH_TOPICS = [
    'clinical presentation, symptoms, and diagnosis criteria',
    'WHO Nigeria treatment guidelines and recommended protocols',
    'NHIS Nigeria coverage and treatment pathways',
    'differential diagnosis and similar conditions to rule out',
    'red flag symptoms requiring immediate emergency care',
    'recommended diagnostic tests and investigations',
    'drug treatment options, dosages, and contraindications',
    'self-care management and home treatment guidelines',
    'nutrition and dietary recommendations during illness',
    'prevention measures and when to seek further care',
]

print(f'âœ… {len(MEDICAL_RESEARCH_TOPICS)} research topics defined')
for i, t in enumerate(MEDICAL_RESEARCH_TOPICS, 1):
    print(f'  {i:2}. {t}')

âœ… 10 research topics defined
   1. clinical presentation, symptoms, and diagnosis criteria
   2. WHO Nigeria treatment guidelines and recommended protocols
   3. NHIS Nigeria coverage and treatment pathways
   4. differential diagnosis and similar conditions to rule out
   5. red flag symptoms requiring immediate emergency care
   6. recommended diagnostic tests and investigations
   7. drug treatment options, dosages, and contraindications
   8. self-care management and home treatment guidelines
   9. nutrition and dietary recommendations during illness
  10. prevention measures and when to seek further care


## 6. Medical Research Node

In [28]:
def medical_research_node(state: MedicalAgentState) -> MedicalAgentState:
    """
    Node: Research symptoms using:
    1. OpenAI web search (WHO + NHIS + clinical guidelines)
    2. PubMed API (peer-reviewed evidence)
    Combines all into raw_research text for embedding.

    Reads:  state['symptoms'], state['raw_message'], state['user_mode']
    Writes: state['raw_research'], state['status']
    """
    symptoms  = state.get('symptoms', [])
    message   = state.get('raw_message', '')
    mode      = state.get('user_mode', 'patient')
    errors    = list(state.get('errors', []))

    # Build symptom query string
    symptom_query = ', '.join(symptoms[:5]) if symptoms else message[:200]
    print(f'[RESEARCH] Researching: "{symptom_query[:60]}"')
    print(f'[RESEARCH] Mode: {mode} | Topics: {len(MEDICAL_RESEARCH_TOPICS)}')

    results = []

    # â”€â”€ Part 1: OpenAI web search across clinical topics â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
    for i, topic in enumerate(MEDICAL_RESEARCH_TOPICS, 1):
        # Build context-aware prompt
        if mode == 'clinician':
            prompt = (
                f'Clinical information for Nigerian healthcare provider:\n'
                f'Condition/symptoms: {symptom_query}\n'
                f'Research topic: {topic}\n'
                f'Focus on: WHO guidelines, NHIS Nigeria protocols, evidence-based medicine.'
                f'Include ICD-10 codes where relevant. Be specific and clinical.'
            )
        else:
            prompt = (
                f'Patient health information for Nigeria context:\n'
                f'Symptoms reported: {symptom_query}\n'
                f'Research topic: {topic}\n'
                f'Focus on: WHO guidelines, NHIS Nigeria, practical guidance.'
                f'Use plain language appropriate for patients.'
            )

        print(f'  [{i}/{len(MEDICAL_RESEARCH_TOPICS)}] {topic[:55]}...')

        for attempt in range(3):
            try:
                response = openai_client.responses.create(
                    model=RESEARCH_MODEL,
                    tools=[{'type': 'web_search_preview'}],
                    input=prompt
                )
                text = ''
                for block in response.output:
                    if hasattr(block, 'content'):
                        for c in block.content:
                            if hasattr(c, 'text'):
                                text += c.text
                if text.strip():
                    results.append(
                        f'### TOPIC: {topic.upper()}\n'
                        f'Source: OpenAI Web Search | Date: {datetime.now(timezone.utc).strftime("%Y-%m-%d")}\n\n'
                        f'{text.strip()}'
                    )
                    break
            except Exception as e:
                print(f'    âš ï¸ Attempt {attempt+1}/3 failed: {e}')
                if attempt == 2:
                    errors.append(f'Topic "{topic[:40]}" failed: {e}')
                else:
                    time.sleep(2 ** attempt)

    # â”€â”€ Part 2: PubMed API â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
    print(f'\n[RESEARCH] Querying PubMed API...')
    pubmed_queries = [
        f'{symptom_query} treatment Nigeria',
        f'{symptom_query} clinical guidelines Africa',
        f'{symptom_query} diagnosis management'
    ]

    pubmed_results = []
    for query in pubmed_queries:
        articles = search_pubmed(query, max_results=3)
        pubmed_results.extend(articles)
        time.sleep(0.5)  # Respect NCBI rate limit

    if pubmed_results:
        pubmed_text = '### PUBMED PEER-REVIEWED EVIDENCE\n'
        pubmed_text += f'Source: PubMed NCBI | Retrieved: {datetime.now(timezone.utc).strftime("%Y-%m-%d")}\n\n'
        for a in pubmed_results[:8]:  # Max 8 articles
            pubmed_text += (
                f'**{a["title"]}**\n'
                f'Authors: {a["authors"]} ({a["year"]}) | Journal: {a["journal"]}\n'
                f'PubMed: {a["url"]}\n\n'
            )
        results.append(pubmed_text)
        print(f'[RESEARCH] âœ… PubMed: {len(pubmed_results)} articles retrieved')
    else:
        print('[RESEARCH] âš ï¸ PubMed returned no results â€” continuing with web search data')

    if not results:
        return {
            **state,
            'status': 'error_research_failed',
            'errors': errors + ['All research sources failed.'],
            'workflow_path': state.get('workflow_path', []) + ['medical_research']
        }

    raw_research = (
        f'# MEDICAL RESEARCH REPORT\n'
        f'Symptoms: {symptom_query}\n'
        f'Mode: {mode} | Generated: {datetime.now(timezone.utc).isoformat()}\n'
        f'Sources: OpenAI Web Search + PubMed NCBI\n\n'
        + '\n\n'.join(results)
    )

    print(f'\n[RESEARCH] âœ… Total: {len(raw_research):,} chars | {len(results)} sections')

    return {
        **state,
        'raw_research': raw_research,
        'status':       'research_complete',
        'errors':       errors,
        'workflow_path': state.get('workflow_path', []) + ['medical_research']
    }

print('âœ… medical_research_node defined')

âœ… medical_research_node defined


## 7. Chunk & Embed Node â€” Clinical-Aware Chunking

In [29]:
def chunk_clinical_text(text: str, chunk_size: int = 200, overlap: int = 30) -> List[Dict]:
    """
    Chunk clinical text with metadata extraction.
    Detects source type (pubmed/who/nhis/web) for each chunk.
    Returns list of dicts with text and metadata.
    """
    words  = text.split()
    step   = chunk_size - overlap
    chunks = []

    for i in range(0, len(words), step):
        chunk_text = ' '.join(words[i:i + chunk_size])
        if not chunk_text.strip():
            continue

        # Detect source type from chunk content
        chunk_lower = chunk_text.lower()
        if 'pubmed' in chunk_lower or 'ncbi' in chunk_lower:
            source_type = 'pubmed'
            evidence_level = 'peer-reviewed'
        elif 'who' in chunk_lower or 'world health' in chunk_lower:
            source_type = 'who_guidelines'
            evidence_level = 'international_guidelines'
        elif 'nhis' in chunk_lower or 'nigeria' in chunk_lower:
            source_type = 'nhis_nigeria'
            evidence_level = 'national_guidelines'
        else:
            source_type = 'web_search'
            evidence_level = 'general'

        chunks.append({
            'text':           chunk_text,
            'chunk_idx':      i // step,
            'source_type':    source_type,
            'evidence_level': evidence_level
        })

    return chunks


def embed_and_store_clinical_node(state: MedicalAgentState) -> MedicalAgentState:
    """
    Node: Chunk clinical research, embed, and upsert to Pinecone.
    Each vector includes rich metadata: source_type, evidence_level, condition.

    Reads:  state['raw_research'], state['symptoms'], state['ticket_id']
    Writes: state['research_chunks'], state['pinecone_ids'], state['status']
    """
    research  = state.get('raw_research', '')
    symptoms  = state.get('symptoms', [])
    ticket    = state.get('ticket_id', 'UNKNOWN')
    mode      = state.get('user_mode', 'patient')
    errors    = list(state.get('errors', []))

    # Namespace by session for isolation
    condition_key = '-'.join(symptoms[0].split()[:3]).lower() if symptoms else 'general'
    condition_key = re.sub(r'[^a-z0-9-]', '', condition_key)[:50]
    namespace     = f'{condition_key}-{ticket[-8:].lower()}'

    print(f'[EMBED] Chunking clinical research...')
    chunks = chunk_clinical_text(research, chunk_size=200, overlap=30)
    print(f'[EMBED] {len(chunks)} chunks created | Namespace: {namespace}')

    vectors      = []
    chunk_ids    = []
    valid_chunks = []

    # Embed in batches
    batch_size = 10
    for batch_start in range(0, len(chunks), batch_size):
        batch      = chunks[batch_start:batch_start + batch_size]
        batch_texts = [c['text'] for c in batch]
        print(f'  Embedding batch {batch_start//batch_size + 1}/{-(-len(chunks)//batch_size)}...')

        for attempt in range(3):
            try:
                response = openai_client.embeddings.create(
                    model=EMBED_MODEL, input=batch_texts
                )
                for j, emb_data in enumerate(response.data):
                    chunk_info = batch[j]
                    chunk_id   = f'{ticket}-c{batch_start+j:04d}'

                    vectors.append({
                        'id':     chunk_id,
                        'values': emb_data.embedding,
                        'metadata': {
                            'ticket_id':      ticket,
                            'condition':      condition_key,
                            'user_mode':      mode,
                            'source_type':    chunk_info['source_type'],
                            'evidence_level': chunk_info['evidence_level'],
                            'chunk_idx':      chunk_info['chunk_idx'],
                            'text':           chunk_info['text'][:1000],
                            'created_at':     datetime.now(timezone.utc).isoformat()
                        }
                    })
                    chunk_ids.append(chunk_id)
                    valid_chunks.append(chunk_info['text'])
                break
            except Exception as e:
                print(f'    âš ï¸ Embed attempt {attempt+1}/3 failed: {e}')
                if attempt == 2:
                    errors.append(f'Embedding batch failed: {e}')
                else:
                    time.sleep(2 ** attempt)

    if not vectors:
        return {
            **state,
            'status': 'error_embed_failed',
            'errors': errors + ['No vectors created.'],
            'workflow_path': state.get('workflow_path', []) + ['embed_and_store']
        }

    # Upsert to Pinecone
    print(f'[EMBED] Upserting {len(vectors)} vectors to Pinecone...')
    for i in range(0, len(vectors), 100):
        pinecone_index.upsert(vectors=vectors[i:i+100], namespace=namespace)

    # Source breakdown
    sources = {}
    for c in chunks:
        sources[c['source_type']] = sources.get(c['source_type'], 0) + 1

    print(f'[EMBED] âœ… {len(vectors)} vectors stored in namespace "{namespace}"')
    print(f'[EMBED]    Source breakdown: {sources}')

    return {
        **state,
        'research_chunks': valid_chunks,
        'pinecone_ids':    chunk_ids,
        'clinical_namespace': namespace,
        'status':          'stored',
        'errors':          errors,
        'workflow_path':   state.get('workflow_path', []) + ['embed_and_store']
    }

print('âœ… embed_and_store_clinical_node defined')

âœ… embed_and_store_clinical_node defined


## 8. Build Sprint 2 Graph

In [30]:
def error_handler_node(state):
    print('\nðŸš¨ PIPELINE ERROR:')
    for e in state.get('errors', []): print(f'  âŒ {e}')
    return {**state, 'status': 'failed',
            'workflow_path': state.get('workflow_path', []) + ['error_handler']}

def route(state):
    return 'error_handler' if 'error' in state.get('status', '') else 'continue'


def build_sprint2_graph():
    g = StateGraph(MedicalAgentState)
    g.add_node('medical_research', medical_research_node)
    g.add_node('embed_and_store',  embed_and_store_clinical_node)
    g.add_node('error_handler',    error_handler_node)

    g.set_entry_point('medical_research')
    g.add_conditional_edges('medical_research', route,
        {'continue': 'embed_and_store', 'error_handler': 'error_handler'})
    g.add_conditional_edges('embed_and_store', route,
        {'continue': END, 'error_handler': 'error_handler'})
    g.add_edge('error_handler', END)
    return g.compile()

sprint2_graph = build_sprint2_graph()
print('âœ… Sprint 2 graph compiled')
print('   Flow: medical_research â†’ embed_and_store â†’ END')

âœ… Sprint 2 graph compiled
   Flow: medical_research â†’ embed_and_store â†’ END


## 9. Run â€” Research a Condition

In [31]:
# â”€â”€ Set the condition to research â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
# Common Nigerian presentations to test:
# 'malaria fever chills headache'
# 'typhoid fever abdominal pain weakness'
# 'hypertension headache dizziness'
# 'diabetes excessive thirst frequent urination'

SYMPTOM_QUERY = 'malaria fever chills headache'  # â† Change to test different conditions

print(f'\n{"="*60}')
print(f'  RESEARCHING: {SYMPTOM_QUERY}')
print(f'{"="*60}')

initial_state = create_medical_state(
    message=f'I have {SYMPTOM_QUERY}',
    symptoms=SYMPTOM_QUERY.split(),
    mode='patient'
)

result = sprint2_graph.invoke(initial_state)

print(f'\n{"="*60}')
print(f'  Status        : {result["status"]}')
print(f'  Chunks created: {len(result.get("research_chunks") or [])}')
print(f'  Vectors stored: {len(result.get("pinecone_ids") or [])}')
print(f'  Workflow path : {" â†’ ".join(result["workflow_path"])}')
print(f'  Errors        : {result.get("errors", [])}')
print(f'{"="*60}')


  RESEARCHING: malaria fever chills headache
[RESEARCH] Researching: "malaria, fever, chills, headache"
[RESEARCH] Mode: patient | Topics: 10
  [1/10] clinical presentation, symptoms, and diagnosis criteria...
  [2/10] WHO Nigeria treatment guidelines and recommended protoc...
  [3/10] NHIS Nigeria coverage and treatment pathways...
  [4/10] differential diagnosis and similar conditions to rule o...
  [5/10] red flag symptoms requiring immediate emergency care...
  [6/10] recommended diagnostic tests and investigations...
  [7/10] drug treatment options, dosages, and contraindications...
  [8/10] self-care management and home treatment guidelines...
  [9/10] nutrition and dietary recommendations during illness...
  [10/10] prevention measures and when to seek further care...

[RESEARCH] Querying PubMed API...
[PUBMED] Found 3 articles for: malaria, fever, chills, headache treatment Nigeria
[PUBMED] Found 1 articles for: malaria, fever, chills, headache clinical guidelin
[PUBMED] No re

In [32]:
# â”€â”€ DoD Verification â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
import pathlib

vectors   = len(result.get('pinecone_ids') or [])
status_ok = result['status'] == 'stored'

print(f'\n{"="*60}')
print('SPRINT 2 â€” DEFINITION OF DONE CHECK')
print(f'{"="*60}')
print(f'{"âœ…" if status_ok else "âŒ"} Status: {result["status"]} (need "stored")')
print(f'{"âœ…" if vectors >= 10 else "âŒ"} Vectors: {vectors} (need â‰¥10)')

# Source breakdown
research = result.get('raw_research', '')
has_pubmed = 'pubmed' in research.lower() or 'ncbi' in research.lower()
has_who    = 'who' in research.lower() or 'world health' in research.lower()
has_nhis   = 'nhis' in research.lower() or 'nigeria' in research.lower()

print(f'{"âœ…" if has_pubmed else "âš ï¸"} PubMed evidence included')
print(f'{"âœ…" if has_who else "âš ï¸"} WHO guidelines included')
print(f'{"âœ…" if has_nhis else "âš ï¸"} Nigeria/NHIS context included')

if status_ok and vectors >= 10:
    print('\nðŸŽ‰ Sprint 2 DoD: MET â€” Ready for Sprint 3')
else:
    print(f'\nâš ï¸ DoD partially met â€” {vectors} vectors stored. Enough to proceed with Sprint 3.')

# Save for Sprint 3
sprint2_output = dict(result)
sprint2_output['research_chunks'] = (sprint2_output.get('research_chunks') or [])[:5]
pathlib.Path('sprint2_medical_output.json').write_text(
    json.dumps(sprint2_output, indent=2, default=str))
print('âœ… Sprint 2 output saved to sprint2_medical_output.json')


SPRINT 2 â€” DEFINITION OF DONE CHECK
âœ… Status: stored (need "stored")
âœ… Vectors: 23 (need â‰¥10)
âœ… PubMed evidence included
âœ… WHO guidelines included
âœ… Nigeria/NHIS context included

ðŸŽ‰ Sprint 2 DoD: MET â€” Ready for Sprint 3
âœ… Sprint 2 output saved to sprint2_medical_output.json


## 10. Bonus â€” Test Multiple Conditions
Run this cell to pre-populate Pinecone with common Nigerian conditions for richer Sprint 3 results.

In [33]:
# â”€â”€ Pre-populate common Nigerian conditions â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
# Uncomment to run â€” takes ~5 minutes but enriches the RAG store significantly

COMMON_CONDITIONS = [
    ('typhoid fever abdominal pain weakness', 'patient'),
    ('hypertension headache dizziness chest pain', 'patient'),
    ('diabetes excessive thirst frequent urination', 'patient'),
    ('malaria treatment protocol Nigeria', 'clinician'),
    ('anaemia weakness fatigue pallor', 'patient'),
]

print('Bonus: Pre-populating Pinecone with common Nigerian conditions')
print('This will take ~5 minutes. Uncomment the loop to run.\n')

# Uncomment to run:
# for condition, mode in COMMON_CONDITIONS:
#     print(f'\n--- Researching: {condition} ({mode}) ---')
#     state = create_medical_state(f'I have {condition}', condition.split(), mode)
#     r = sprint2_graph.invoke(state)
#     print(f'    Status: {r["status"]} | Vectors: {len(r.get("pinecone_ids") or [])}')
#     time.sleep(2)  # Rate limit
# print('\nâœ… All conditions pre-populated in Pinecone')

print('Ready for Sprint 3 â€” RAG Retrieval + Cohere Reranking + LangGraph Triage')

Bonus: Pre-populating Pinecone with common Nigerian conditions
This will take ~5 minutes. Uncomment the loop to run.

Ready for Sprint 3 â€” RAG Retrieval + Cohere Reranking + LangGraph Triage
